In [42]:
import os
import pandas as pd
from bs4 import BeautifulSoup
from io import StringIO
SCORE_DIR = "data2/scores"

In [45]:
def parse_html(box_score):
    with open(box_score) as f: 
        html = f.read()
        
    soup = BeautifulSoup(html)
    # selects the pieces that needs to be removed from the data
    # decompose = removes all the selected pieces from the html
    [s.decompose() for s in soup.select("tr.over_header")]
    [s.decompose() for s in soup.select("tr.thead")]
    
    return soup

In [46]:
def read_line_score(soup):
    # attrs = attributes, functions like selector for beautifulSoup but for pandas
    # read_html by default returns a list of data frame
    # StringIO is needed because if not pandas with lxml will interpret 
    # str(soup) as a filename/URL instead of as an html content 
    line_score = pd.read_html(StringIO(str(soup)), attrs={"id": "line_score"})[0]
    cols = list(line_score.columns)
    cols[0] = "team"
    cols[-1] = "total"
    line_score.columns = cols
    
    line_score = line_score[["team", "total"]]
    return line_score
    

In [47]:
def read_stats(soup, team, stat):
    
    # index_col => the player's name
    df = pd.read_html(StringIO(str(soup)), attrs={"id": f"box-{team}-game-{stat}"}, index_col=0)[0]
    # pd.to_numeric => removes the data in STRING such as DID NOT PLAY into 
    # numeric value NAN, meaning missing value, so the machine can take it for \
    # machine learning
    df = df.apply(pd.to_numeric, errors="coerce")
    return df

In [48]:
def read_season_info(soup):
    # select by default returns a list, we simply want the first element
    nav = soup.select("#bottom_nav_container")[0]
    hrefs = [a["href"] for a in nav.find_all("a")]
    season = os.path.basename(hrefs[1]).split("_")[0]
    
    return season

In [49]:
box_scores = os.listdir(SCORE_DIR)
box_scores = [os.path.join(SCORE_DIR, f) for f in box_scores if f.endswith("html")]



In [65]:
base_cols = None
games = []

for box_score in box_scores:
     
     soup = parse_html(box_score)
     print(box_score)
     line_score = read_line_score(soup)
     teams = list(line_score["team"])
     print(line_score.columns.get_loc("team"))
     summaries = []
     for team in teams:
          
          basic = read_stats(soup, team, "basic")
          advanced = read_stats(soup, team, "advanced")
          
          # Extract the total data stats for each team as for in order for machines
          # to learn/take data, the data needs to be in a single row
          # iloc method indexes data frame by position, in our case -1 means to take
          # the last row and the : means to take all rows
          total = pd.concat([basic.iloc[-1, :], advanced.iloc[-1, :]])
          total.index = total.index.str.lower()
          
          # For this it will mean to look through all rows excpet the last one
          maxes = pd.concat([basic.iloc[:-1, :].max(), advanced.iloc[:-1, :].max()])
          maxes.index = maxes.index.str.lower() + "_max"

          summary = pd.concat([total, maxes])
          
          if base_cols is None:
               # First time it loops through the loop, it will go over the html and 
               # find all the values in the box_score and determine those values 
               # to look for in other box_scores as well. Also removes any duplicate
               # value, for ex: MP(minutes played) exists in both advanced and basic
               base_cols = list(summary.index.drop_duplicates(keep="first"))
               base_cols = [b for b in base_cols if "bpm" not in b]
          
          summary = summary[base_cols]
          
          summaries.append(summary)

     summary = pd.concat(summaries, axis=1).T

     game = pd.concat([summary, line_score], axis = 1)
     
     print(f"IN game team is located at: {game.columns.get_loc("team")}")
     

     game["home"] = [0, 1]

     # Flips the data frame so the first and second row switches
     game_opp = game.iloc[::-1].reset_index()
     game_opp.columns += "_opp"

     full_game = pd.concat([game, game_opp], axis = 1)
     full_game["season"] = read_season_info(soup)
     full_game["date"] = os.path.basename(box_score)[:8]
     full_game["date"] = pd.to_datetime(full_game["date"], format = "%Y%m%d")
     full_game["won"] = full_game["total"] > full_game["total_opp"]

     print(f"IN full_game team is located at: {full_game.columns.get_loc("team")}")

     games.append(full_game)
     
     print(f"IN games team is located at: {games[0].columns.get_loc("team")}")

     if len(games) % 100 == 0:
          print(f"{len(games)} / {len(box_scores)}")

data2/scores/202211070MIA.html
0
IN game team is located at: 72
IN full_game team is located at: 72
IN games team is located at: 72
data2/scores/202212140DAL.html
0
IN game team is located at: 72
IN full_game team is located at: 72
IN games team is located at: 72
data2/scores/202212070TOR.html
0
IN game team is located at: 72
IN full_game team is located at: 72
IN games team is located at: 72
data2/scores/202501290GSW.html
0


KeyboardInterrupt: 

In [73]:
games_df.columns.get_loc("team")

72

In [40]:
# Combine all the games and treat each game as rows
games_df = pd.concat(games, ignore_index=True)

In [66]:
games_df

,mp,mp,fg,fga,fg%,3p,3pa,3p%,ft,fta,...,tov%_max_opp,usg%_max_opp,ortg_max_opp,drtg_max_opp,team_opp,total_opp,home_opp,season,date,won
0,240.0,240.0,38.0,72.0,0.528,16.0,37.0,0.432,18.0,21.0,...,26.2,26.8,155.0,123.0,MIA,107,1,2023,2022-11-07,True
1,240.0,240.0,40.0,84.0,0.476,14.0,39.0,0.359,13.0,15.0,...,41.0,37.3,160.0,121.0,POR,110,0,2023,2022-11-07,False
2,240.0,240.0,41.0,78.0,0.526,8.0,24.0,0.333,15.0,19.0,...,28.6,41.1,250.0,125.0,DAL,90,1,2023,2022-12-14,True
3,240.0,240.0,29.0,74.0,0.392,13.0,38.0,0.342,19.0,26.0,...,12.6,33.0,183.0,110.0,CLE,105,0,2023,2022-12-14,False
4,240.0,240.0,37.0,87.0,0.425,7.0,33.0,0.212,32.0,35.0,...,20.0,32.3,226.0,117.0,TOR,126,1,2023,2022-12-07,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17647,240.0,240.0,33.0,85.0,0.388,12.0,44.0,0.273,28.0,34.0,...,51.5,36.2,141.0,114.0,BOS,117,0,2020,2020-09-19,False
17648,240.0,240.0,43.0,91.0,0.473,17.0,45.0,0.378,23.0,32.0,...,53.2,31.1,225.0,130.0,UTA,124,1,2025,2024-10-23,True
17649,240.0,240.0,40.0,85.0,0.471,10.0,26.0,0.385,34.0,45.0,...,22.5,34.2,258.0,125.0,MEM,126,0,2025,2024-10-23,False
17650,240.0,240.0,42.0,82.0,0.512,12.0,27.0,0.444,12.0,19.0,...,53.2,28.0,146.0,126.0,TOR,121,1,2024,2023-11-24,False


In [59]:
games_df.to_csv("nba_games.csv")

In [9]:
box_scores

['data2/scores/202211070MIA.html',
 'data2/scores/202212140DAL.html',
 'data2/scores/202212070TOR.html',
 'data2/scores/202501290GSW.html',
 'data2/scores/202604020LAC.html',
 'data2/scores/202104300MEM.html',
 'data2/scores/202402130BRK.html',
 'data2/scores/202501280GSW.html',
 'data2/scores/202311110ATL.html',
 'data2/scores/202412110HOU.html',
 'data2/scores/202212090UTA.html',
 'data2/scores/202404110POR.html',
 'data2/scores/202201230MIN.html',
 'data2/scores/202001160NOP.html',
 'data2/scores/202203180PHO.html',
 'data2/scores/202102190ORL.html',
 'data2/scores/202411080MEM.html',
 'data2/scores/202603180IND.html',
 'data2/scores/201911180HOU.html',
 'data2/scores/202210290BRK.html',
 'data2/scores/202411200CLE.html',
 'data2/scores/202605070OKC.html',
 'data2/scores/202603110ORL.html',
 'data2/scores/202111240GSW.html',
 'data2/scores/202310280CLE.html',
 'data2/scores/202103030TOR.html',
 'data2/scores/202411080DEN.html',
 'data2/scores/202211070DET.html',
 'data2/scores/20210

In [58]:
advanced

,MP,TS%,eFG%,3PAr,FTr,ORB%,DRB%,TRB%,AST%,STL%,BLK%,TOV%,USG%,ORtg,DRtg,BPM
Starters,,,,,,,,,,,,,,,,
Kyle Lowry,NaN,0.889,0.875,0.750,0.125,0.0,31.2,14.0,33.4,0.0,0.0,26.2,15.7,135.0,123.0,2.8
Caleb Martin,NaN,0.455,0.455,0.455,0.000,10.9,8.9,10.0,4.2,1.5,0.0,0.0,15.1,107.0,121.0,-6.1
Bam Adebayo,NaN,0.554,0.571,0.000,0.071,7.6,13.9,10.4,15.1,1.5,4.1,6.5,22.1,112.0,118.0,-1.4
Jimmy Butler,NaN,0.481,0.357,0.071,0.429,0.0,18.6,8.4,30.8,9.2,0.0,10.7,26.8,104.0,102.0,4.4
Max Strus,NaN,0.800,0.800,0.800,0.000,0.0,0.0,0.0,13.3,2.0,0.0,0.0,18.9,148.0,120.0,5.3
Gabe Vincent,NaN,0.524,0.500,0.692,0.231,0.0,4.4,2.0,12.2,2.9,3.9,17.3,23.3,89.0,116.0,-5.2
Duncan Robinson,NaN,0.500,0.500,1.000,0.000,0.0,21.7,9.8,14.4,0.0,0.0,10.0,16.7,96.0,123.0,-6.0
Dewayne Dedmon,NaN,0.740,0.600,0.200,0.800,0.0,63.6,28.6,10.9,0.0,0.0,0.0,22.1,155.0,120.0,7.2
Haywood Highsmith,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [66]:
total

mp      240.000
fg       40.000
fga      84.000
fg%       0.476
3p       14.000
3pa      39.000
3p%       0.359
ft       13.000
fta      15.000
ft%       0.867
orb       5.000
drb      27.000
trb      32.000
ast      28.000
stl      11.000
blk       2.000
tov      12.000
pf       18.000
pts     107.000
gmsc        NaN
+/-         NaN
mp      240.000
ts%       0.591
efg%      0.560
3par      0.464
ftr       0.179
orb%     13.200
drb%     87.100
trb%     46.400
ast%     70.000
stl%     11.500
blk%      5.700
tov%     11.700
usg%    100.000
ortg    112.300
drtg    115.500
bpm         NaN
Name: Team Totals, dtype: float64

In [67]:
maxes

mp_max          NaN
fg_max        8.000
fga_max      14.000
fg%_max       0.625
3p_max        4.000
3pa_max       9.000
3p%_max       0.667
ft_max        6.000
fta_max       6.000
ft%_max       1.000
orb_max       3.000
drb_max       7.000
trb_max       7.000
ast_max       8.000
stl_max       6.000
blk_max       1.000
tov_max       3.000
pf_max        5.000
pts_max      16.000
gmsc_max     17.100
+/-_max       2.000
mp_max          NaN
ts%_max       0.889
efg%_max      0.875
3par_max      1.000
ftr_max       0.800
orb%_max     10.900
drb%_max     63.600
trb%_max     28.600
ast%_max     33.400
stl%_max      9.200
blk%_max      4.100
tov%_max     26.200
usg%_max     26.800
ortg_max    155.000
drtg_max    123.000
bpm_max       7.200
dtype: float64

In [77]:
summary

,mp,mp,fg,fga,fg%,3p,3pa,3p%,ft,fta,...,orb%_max,drb%_max,trb%_max,ast%_max,stl%_max,blk%_max,tov%_max,usg%_max,ortg_max,drtg_max
0,240.0,240.0,38.0,72.0,0.528,16.0,37.0,0.432,18.0,21.0,...,12.9,44.1,26.7,30.6,3.9,8.1,41.0,37.3,160.0,121.0
1,240.0,240.0,40.0,84.0,0.476,14.0,39.0,0.359,13.0,15.0,...,10.9,63.6,28.6,33.4,9.2,4.1,26.2,26.8,155.0,123.0


In [55]:
games_df

,mp,mp,fg,fga,fg%,3p,3pa,3p%,ft,fta,...,tov%_max_opp,usg%_max_opp,ortg_max_opp,drtg_max_opp,team_opp,total_opp,home_opp,season,date,won
0,240.0,240.0,38.0,72.0,0.528,16.0,37.0,0.432,18.0,21.0,...,26.2,26.8,155.0,123.0,MIA,107,1,2023,2022-11-07,True
1,240.0,240.0,40.0,84.0,0.476,14.0,39.0,0.359,13.0,15.0,...,41.0,37.3,160.0,121.0,POR,110,0,2023,2022-11-07,False
2,240.0,240.0,41.0,78.0,0.526,8.0,24.0,0.333,15.0,19.0,...,28.6,41.1,250.0,125.0,DAL,90,1,2023,2022-12-14,True
3,240.0,240.0,29.0,74.0,0.392,13.0,38.0,0.342,19.0,26.0,...,12.6,33.0,183.0,110.0,CLE,105,0,2023,2022-12-14,False
4,240.0,240.0,37.0,87.0,0.425,7.0,33.0,0.212,32.0,35.0,...,20.0,32.3,226.0,117.0,TOR,126,1,2023,2022-12-07,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17647,240.0,240.0,33.0,85.0,0.388,12.0,44.0,0.273,28.0,34.0,...,51.5,36.2,141.0,114.0,BOS,117,0,2020,2020-09-19,False
17648,240.0,240.0,43.0,91.0,0.473,17.0,45.0,0.378,23.0,32.0,...,53.2,31.1,225.0,130.0,UTA,124,1,2025,2024-10-23,True
17649,240.0,240.0,40.0,85.0,0.471,10.0,26.0,0.385,34.0,45.0,...,22.5,34.2,258.0,125.0,MEM,126,0,2025,2024-10-23,False
17650,240.0,240.0,42.0,82.0,0.512,12.0,27.0,0.444,12.0,19.0,...,53.2,28.0,146.0,126.0,TOR,121,1,2024,2023-11-24,False


In [113]:
game_opp

,mp_opp,mp_opp,fg_opp,fga_opp,fg%_opp,3p_opp,3pa_opp,3p%_opp,ft_opp,fta_opp,...,ast%_max_opp,stl%_max_opp,blk%_max_opp,tov%_max_opp,usg%_max_opp,ortg_max_opp,drtg_max_opp,team_opp,total_opp,home_opp
1,240.0,240.0,40.0,84.0,0.476,14.0,39.0,0.359,13.0,15.0,...,33.4,9.2,4.1,26.2,26.8,155.0,123.0,MIA,107,1
0,240.0,240.0,38.0,72.0,0.528,16.0,37.0,0.432,18.0,21.0,...,30.6,3.9,8.1,41.0,37.3,160.0,121.0,POR,110,0


In [38]:
full_game

,mp,mp,fg,fga,fg%,3p,3pa,3p%,ft,fta,...,tov%_max_opp,usg%_max_opp,ortg_max_opp,drtg_max_opp,team_opp,total_opp,home_opp,season,date,won
0,240.0,240.0,42.0,82.0,0.512,12.0,27.0,0.444,12.0,19.0,...,53.2,28.0,146.0,126.0,TOR,121,1,2024,2023-11-24,False
1,240.0,240.0,44.0,83.0,0.530,13.0,39.0,0.333,20.0,27.0,...,25.0,34.3,195.0,136.0,CHI,108,0,2024,2023-11-24,True


In [39]:
games_df = pd.concat(games, ignore_index=)

[      mp     mp    fg   fga    fg%    3p   3pa    3p%    ft   fta  ...  \
 0  240.0  240.0  38.0  72.0  0.528  16.0  37.0  0.432  18.0  21.0  ...   
 1  240.0  240.0  40.0  84.0  0.476  14.0  39.0  0.359  13.0  15.0  ...   
 
    tov%_max_opp  usg%_max_opp  ortg_max_opp  drtg_max_opp  team_opp  \
 0          26.2          26.8         155.0         123.0       MIA   
 1          41.0          37.3         160.0         121.0       POR   
 
    total_opp  home_opp  season       date    won  
 0        107         1    2023 2022-11-07   True  
 1        110         0    2023 2022-11-07  False  
 
 [2 rows x 154 columns],
       mp     mp    fg   fga    fg%    3p   3pa    3p%    ft   fta  ...  \
 0  240.0  240.0  41.0  78.0  0.526   8.0  24.0  0.333  15.0  19.0  ...   
 1  240.0  240.0  29.0  74.0  0.392  13.0  38.0  0.342  19.0  26.0  ...   
 
    tov%_max_opp  usg%_max_opp  ortg_max_opp  drtg_max_opp  team_opp  \
 0          28.6          41.1         250.0         125.0       DAL   
 